# Alarm Lead Time Viewer

Use this notebook to inspect how many days before each target event the first valid true alarm occurs. Select one method and one setting, then click the button. Files are loaded only after selection.

- SCC uses the alarm episode start time, which corresponds to the point where the `min_consecutive` rule first becomes an alarm.
- CUSUM uses the start of each continuous alarm episode (its first alarm point); an episode is valid if it starts before the event and reaches the seven-day horizon.
- BCAD uses the alarm episode start time, which is the sixth consecutive anomaly point under the fixed `c=6` rule.

In [8]:
from pathlib import Path
import json
import re
import subprocess
import sys

import pandas as pd
from IPython.display import Image, Markdown, clear_output, display

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXPERIMENTS_DIR = PROJECT_ROOT / "results" / "kelmarsh" / "experiments"
DEFAULT_RUN_ID = "all6_multi3_seq12_h64_l1_bs64_lr1e-03_wd0_do0_e60_p8_seed42"
METHODS = ["scc", "cusum", "bcad"]

def list_experiments():
    if not EXPERIMENTS_DIR.exists():
        return [DEFAULT_RUN_ID]
    runs = sorted(p.name for p in EXPERIMENTS_DIR.iterdir() if (p / "metadata.json").exists())
    return runs or [DEFAULT_RUN_ID]

def format_number(value):
    return f"{value:g}".replace(".", "p")

def signed_scc_settings_from_grids(result_dir):
    settings = []
    for grid_path in sorted(result_dir.glob("scc_grid_signed_meanstd_abslimit*.csv")):
        try:
            grid = pd.read_csv(grid_path, usecols=["k", "min_consecutive"])
        except Exception:
            continue
        for _, row in grid.drop_duplicates(["k", "min_consecutive"]).iterrows():
            c = int(row["min_consecutive"])
            value = f"signed_meanstd_abslimit_k{format_number(float(row['k']))}_c{c}"
            settings.append((f"signed SCC: k={float(row['k']):g}, c={c}", value))
    return settings

def cusum_buffer6h_settings_from_grid(result_dir):
    grid_path = result_dir / "cusum_grid_fixed_meanstd_no_consecutive_k0-1_h20-200_buffer6h.csv"
    if not grid_path.exists():
        return []
    grid = pd.read_csv(grid_path, usecols=["k", "h"])
    settings = []
    for _, row in grid.drop_duplicates(["k", "h"]).sort_values(["k", "h"]).iterrows():
        k = float(row["k"])
        h = float(row["h"])
        value = f"twosided_std_k{format_number(k)}_h{format_number(h)}_reset"
        settings.append((f"k={k:g}, h={h:g}", value))
    return settings

def list_settings(run_id, method):
    result_dir = EXPERIMENTS_DIR / run_id
    if not result_dir.exists():
        return []
    if method == "scc":
        prefix = "scc_alarm_episodes_"
        settings = []
        for path in sorted(result_dir.glob(f"{prefix}*.csv")):
            value = path.name[len(prefix):-4]
            label = value
            settings.append((label, value))
        settings.extend(signed_scc_settings_from_grids(result_dir))
        seen = {}
        for label, value in settings:
            seen[value] = label
        return [(label, value) for value, label in sorted(seen.items(), key=lambda item: item[1])]
    if method == "cusum":
        return cusum_buffer6h_settings_from_grid(result_dir)
    prefix = f"{method}_detections_"
    settings = []
    for path in sorted(result_dir.glob(f"{prefix}*.csv")):
        value = path.name[len(prefix):-4]
        if method == "bcad" and "_aw" in value:
            continue
        settings.append((value, value))
    return settings

def run_lead_time_analysis(run_id, method, setting):
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "analysis" / "main_analyze_alarm_lead_time.py"),
        "--run-id", run_id,
        "--method", method,
        "--setting", setting,
    ]
    result = subprocess.run(cmd, cwd=PROJECT_ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError("Lead-time analysis failed")

    csv_path = None
    json_path = None
    fig_path = None
    for line in result.stdout.splitlines():
        if line.startswith("Lead-time CSV saved to:"):
            csv_path = Path(line.split(":", 1)[1].strip())
        elif line.startswith("Summary JSON saved to:"):
            json_path = Path(line.split(":", 1)[1].strip())
        elif line.startswith("Figure saved to:"):
            fig_path = Path(line.split(":", 1)[1].strip())

    if json_path and json_path.exists():
        summary = json.loads(json_path.read_text(encoding="utf-8"))
        keep = [
            "method", "setting", "total_events", "events_with_valid_ta_alarm",
            "events_without_valid_ta_alarm", "detection_rate_among_events",
            "lead_days_mean", "lead_days_50%", "lead_days_max",
        ]
        rows = [(k, summary.get(k)) for k in keep if k in summary]
        display(pd.DataFrame(rows, columns=["metric", "value"]))
    if csv_path and csv_path.exists():
        lead_times = pd.read_csv(csv_path)
        display(Markdown(f"Lead-time CSV: `{csv_path}`"))
        display(lead_times.sort_values("lead_days", ascending=False).head(15))
    if fig_path and fig_path.exists():
        display(Image(filename=str(fig_path)))
    return csv_path, fig_path


In [9]:
if HAS_WIDGETS:
    run_dropdown = widgets.Dropdown(options=list_experiments(), value=DEFAULT_RUN_ID, description="Run")
    method_dropdown = widgets.Dropdown(options=METHODS, value="scc", description="Method")
    settings = list_settings(DEFAULT_RUN_ID, "scc")
    setting_dropdown = widgets.Dropdown(options=settings, value=settings[0][1] if settings else None, description="Setting")
    button = widgets.Button(description="Run lead-time analysis", button_style="primary")
    output = widgets.Output()

    def refresh_settings(*_):
        settings = list_settings(run_dropdown.value, method_dropdown.value)
        setting_dropdown.options = settings
        setting_dropdown.value = settings[0][1] if settings else None

    def on_click(_):
        with output:
            clear_output(wait=True)
            if not setting_dropdown.value:
                print("No setting found for this method/run.")
                return
            run_lead_time_analysis(run_dropdown.value, method_dropdown.value, setting_dropdown.value)

    run_dropdown.observe(refresh_settings, names="value")
    method_dropdown.observe(refresh_settings, names="value")
    button.on_click(on_click)
    display(widgets.VBox([
        widgets.HBox([run_dropdown]),
        widgets.HBox([method_dropdown, setting_dropdown]),
        button,
        output,
    ]))
else:
    print("ipywidgets is not available. Use run_lead_time_analysis(run_id, method, setting) manually.")


In [10]:
# Manual example if widgets are unavailable:
# run_lead_time_analysis(
#     DEFAULT_RUN_ID,
#     method="scc",
#     setting="abs_meanstd_k3_c6",
# )
